# Training Run Provenance

**Note:** Transformer training was reproduced in Google Colab with a GPU runtime. This notebook includes executed outputs for standard RoBERTa and weighted-loss RoBERTa training/evaluation.

- Weighted RoBERTa validation F1: 0.5740
- Weighted RoBERTa test F1: 0.5444
- Metric summary: ../reports/metrics.md


In [1]:
import torch

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [2]:
!pip install transformers datasets evaluate accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00


In [3]:
# IMPORTS
import pandas as pd
import numpy as np
import evaluate

from datasets import Dataset
from transformers import (
    DistilBertTokenizer,
    DistilBertForSequenceClassification,
    RobertaTokenizer,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer
)

In [4]:
from google.colab import files
uploaded = files.upload()

Saving test.tsv to test.tsv
Saving train.tsv to train.tsv
Saving valid.tsv to valid.tsv


In [5]:
cols = [
    "id", "label", "statement", "subject", "speaker", "speaker_job",
    "state", "party", "barely_true_counts", "false_counts",
    "half_true_counts",
    "mostly_true_counts", "pants_fire_counts", "context"
]

train_df = pd.read_csv("train.tsv", sep="\t", header=None, names=cols)
valid_df = pd.read_csv("valid.tsv", sep="\t", header=None, names=cols)
test_df = pd.read_csv("test.tsv", sep="\t", header=None, names=cols)

In [6]:
train_data = train_df[["statement", "label"]].copy()
valid_data = valid_df[["statement", "label"]].copy()
test_data = test_df[["statement", "label"]].copy()

In [7]:
def convert_label(label):
    if label in ["true", "mostly-true"]:
        return "REAL"
    else:
        return "FAKE"

In [8]:
for data in [train_data, valid_data, test_data]:
    data["target"] = data["label"].apply(convert_label)
    data["target_encoded"] = data["target"].map({
        "FAKE": 0,
        "REAL": 1
    })

print(train_data["target"].value_counts())

target
FAKE    6602
REAL    3638
Name: count, dtype: int64


In [9]:
train_hf = Dataset.from_pandas(train_data[["statement", "target_encoded"]])
valid_hf = Dataset.from_pandas(valid_data[["statement", "target_encoded"]])
test_hf = Dataset.from_pandas(test_data[["statement", "target_encoded"]])

In [10]:
accuracy = evaluate.load("accuracy")
precision = evaluate.load("precision")
recall = evaluate.load("recall")
f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy.compute(predictions=predictions, references=labels)["accuracy"],
        "precision": precision.compute(predictions=predictions, references=labels)["precision"],
        "recall": recall.compute(predictions=predictions, references=labels)["recall"],
        "f1": f1.compute(predictions=predictions, references=labels)["f1"],
    }

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


# DistilBert

In [ ]:
distilbert_tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

distilbert_model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

In [ ]:
def distilbert_tokenize(batch):
    return distilbert_tokenizer(
        batch["statement"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

distilbert_train_hf = train_hf.map(distilbert_tokenize, batched=True)
distilbert_valid_hf = valid_hf.map(distilbert_tokenize, batched=True)
distilbert_test_hf = test_hf.map(distilbert_tokenize, batched=True)

In [ ]:
distilbert_train_hf = distilbert_train_hf.rename_column("target_encoded", "labels")
distilbert_valid_hf = distilbert_valid_hf.rename_column("target_encoded", "labels")
distilbert_test_hf = distilbert_test_hf.rename_column("target_encoded", "labels")

In [ ]:
distilbert_train_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
distilbert_valid_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
distilbert_test_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:
distilbert_training_args = TrainingArguments(
    output_dir="./distilbert_fake_news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=2,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [ ]:
distilbert_trainer = Trainer(
    model=distilbert_model,
    args=distilbert_training_args,
    train_dataset=distilbert_train_hf,
    eval_dataset=distilbert_valid_hf,
    processing_class=distilbert_tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
# !pip uninstall -y torchvision

In [ ]:
distilbert_trainer.train()

In [ ]:
# DISTILBERT VALIDATION RESULT

distilbert_results = distilbert_trainer.evaluate(distilbert_valid_hf)
print(distilbert_results)

# RoBERTa

In [12]:
roberta_tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

roberta_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [13]:
def roberta_tokenize(batch):
    return roberta_tokenizer(
        batch["statement"],
        padding="max_length",
        truncation=True,
        max_length=256
    )
roberta_train_hf = train_hf.map(roberta_tokenize, batched=True)
roberta_valid_hf = valid_hf.map(roberta_tokenize, batched=True)
roberta_test_hf =  test_hf.map(roberta_tokenize, batched=True)

Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Map:   0%|          | 0/1284 [00:00<?, ? examples/s]

Map:   0%|          | 0/1267 [00:00<?, ? examples/s]

In [14]:
roberta_train_hf = roberta_train_hf.rename_column("target_encoded", "labels")
roberta_valid_hf = roberta_valid_hf.rename_column("target_encoded", "labels")
roberta_test_hf = roberta_test_hf.rename_column("target_encoded", "labels")

In [15]:
roberta_train_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
roberta_valid_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
roberta_test_hf.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

In [16]:
roberta_training_args = TrainingArguments(
    output_dir="./roberta_fake_news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

In [17]:
roberta_trainer = Trainer(
    model=roberta_model,
    args=roberta_training_args,
    train_dataset=roberta_train_hf,
    eval_dataset=roberta_valid_hf,
    processing_class=roberta_tokenizer,
    compute_metrics=compute_metrics
)

In [ ]:
roberta_trainer.train()

In [ ]:
roberta_results = roberta_trainer.evaluate(roberta_valid_hf)
print(roberta_results)

In [ ]:
roberta_test_results = roberta_trainer.evaluate(roberta_test_hf)
print(roberta_test_results)

In [ ]:
#  SAVE FINAL ROBERTA MODEL

roberta_trainer.save_model("./final_roberta_fake_news")
roberta_tokenizer.save_pretrained("./final_roberta_fake_news")

In [ ]:
!zip -r final_roberta_fake_news.zip final_roberta_fake_news

from google.colab import files
files.download("final_roberta_fake_news.zip")

# Weighted RoBERTa Training

In [18]:
import random
from torch import nn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

CLASS_WEIGHTS = [1.0, 1.8]  # FAKE=0, REAL=1

def compute_metrics_with_auc(eval_pred):
    logits = eval_pred.predictions
    labels = eval_pred.label_ids

    predictions = np.argmax(logits, axis=1)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()[:, 1]

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "recall": recall_score(labels, predictions, zero_division=0),
        "f1": f1_score(labels, predictions, zero_division=0),
        "roc_auc": roc_auc_score(labels, probs),
    }

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        class_weights = torch.tensor(
            CLASS_WEIGHTS,
            dtype=torch.float,
            device=logits.device
        )

        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)

        return (loss, outputs) if return_outputs else loss

In [19]:
weighted_roberta_model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2,
    id2label={0: "FAKE", 1: "REAL"},
    label2id={"FAKE": 0, "REAL": 1}
)

weighted_roberta_training_args = TrainingArguments(
    output_dir="./weighted_roberta_fake_news",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
    data_seed=SEED
)

weighted_roberta_trainer = WeightedTrainer(
    model=weighted_roberta_model,
    args=weighted_roberta_training_args,
    train_dataset=roberta_train_hf,
    eval_dataset=roberta_valid_hf,
    processing_class=roberta_tokenizer,
    compute_metrics=compute_metrics_with_auc
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [11]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
tokenizer=roberta_tokenizer,

In [20]:
import sys
import datasets.config as datasets_config

datasets_config.TORCHVISION_AVAILABLE = False

for name in list(sys.modules):
    if name.startswith("torchvision"):
        del sys.modules[name]

print("Disabled torchvision inside datasets.")

Disabled torchvision inside datasets.


In [21]:
weighted_roberta_trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.645934,0.647675,0.630062,0.453782,0.642857,0.532020,0.687464
2,0.629292,0.613246,0.671340,0.498127,0.633333,0.557652,0.722600
3,0.429668,0.702183,0.668224,0.494828,0.683333,0.574000,0.720789


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3840, training_loss=0.597870247066021, metrics={'train_runtime': 1458.8332, 'train_samples_per_second': 21.058, 'train_steps_per_second': 2.632, 'total_flos': 4041385810329600.0, 'train_loss': 0.597870247066021, 'epoch': 3.0})

In [22]:
weighted_valid_results = weighted_roberta_trainer.evaluate(roberta_valid_hf)
print("Weighted RoBERTa validation results:")
print(weighted_valid_results)

weighted_test_results = weighted_roberta_trainer.evaluate(roberta_test_hf)
print("Weighted RoBERTa test results:")
print(weighted_test_results)

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.429668,0.702183,3,0.668224,0.494828,0.683333,0.574000,0.720789


Weighted RoBERTa validation results:
{'eval_loss': 0.7021832466125488, 'eval_accuracy': 0.6682242990654206, 'eval_precision': 0.49482758620689654, 'eval_recall': 0.6833333333333333, 'eval_f1': 0.574, 'eval_roc_auc': 0.7207892416225748}


Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1,Roc Auc
0.429668,0.753780,3,0.615627,0.469355,0.648107,0.544434,0.684003


Weighted RoBERTa test results:
{'eval_loss': 0.7537800669670105, 'eval_accuracy': 0.6156274664561957, 'eval_precision': 0.4693548387096774, 'eval_recall': 0.6481069042316259, 'eval_f1': 0.5444340505144996, 'eval_roc_auc': 0.6840030276463317}


In [23]:
weighted_roberta_trainer.save_model("./final_roberta_fake_news")
roberta_tokenizer.save_pretrained("./final_roberta_fake_news")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./final_roberta_fake_news/tokenizer_config.json',
 './final_roberta_fake_news/tokenizer.json')

In [24]:
!zip -r final_roberta_fake_news.zip final_roberta_fake_news

from google.colab import files
files.download("final_roberta_fake_news.zip")

  adding: final_roberta_fake_news/ (stored 0%)
  adding: final_roberta_fake_news/config.json (deflated 51%)
  adding: final_roberta_fake_news/training_args.bin (deflated 53%)
  adding: final_roberta_fake_news/tokenizer_config.json (deflated 51%)
  adding: final_roberta_fake_news/tokenizer.json (deflated 82%)
  adding: final_roberta_fake_news/model.safetensors (deflated 12%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>